# Step 6 — Model Inference & Demo

This notebook demonstrates how the clinical intelligence distilled from our fine-tuning process is run in production. We showcase **three core features** plus an **interactive chatbot session**.

## Features Overview
1. **Estimated Insulin Need**: Calculates corrective and meal boluses based on deduced ICR (insulin-to-carb ratio) and ISF (insulin sensitivity factor).
2. **Meal Impact Estimator**: Analyzes historical spikes to recommend bolus strategies (e.g. split bolusing for high-fat meals).
3. **Health Insight (Reading Mode)**: Empathetic, context-aware analysis of recent readings compared to targets.
4. **Diabetic Assistant Chatbot**: A multi-turn conversation partner with explicit medical guardrails (like the 15-15 rule for hypos).

## 1. Setup & Configuration

We use the `google-genai` SDK. Ensure your environment variable `GEMINI_API_KEY` is set.

In [ ]:
import os
import json
from datetime import datetime
from google import genai
from google.genai import types

# Read API key from environment variable
API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
if API_KEY == "YOUR_API_KEY_HERE" or not API_KEY:
    print("⚠️ Warning: GEMINI_API_KEY environment variable not set. Please set it or enter your key.")

client = genai.Client(api_key=API_KEY if API_KEY != "YOUR_API_KEY_HERE" else None)

# Patient profile metadata
DIABETES_TYPE = "Type 1"
REGIMEN = "MDI (Novolog + Tresiba)"
TARGET_RANGE = (70, 140)  # mg/dL
CORRECTION_TARGET = 110   # mg/dL

## 2. Sample 10-Day Glucose Log

This is the compact log format representing the patient's history.

In [ ]:
LOGS_10D = """
--- DAY -9 ---
[08:20] Glucose: 112
[08:35] Meal: Omelet with spinach & feta (~5g carbs)
[08:35] Insulin: 0.5u rapid
[10:35] Glucose: 114
[13:00] Glucose: 102
[13:10] Meal: Mixed greens with tuna & olive oil (~8g carbs)
[13:10] Insulin: 1u rapid
[18:30] Glucose: 98
[18:45] Meal: Deep dish pepperoni pizza (~85g carbs, high fat)
[18:45] Insulin: 8u rapid
[20:45] Glucose: 125
[22:45] Glucose: 198 (Delayed spike)

--- DAY -8 ---
[08:10] Glucose: 118
[08:20] Meal: Oatmeal + banana (~55g carbs)
[08:20] Insulin: 5u rapid
[10:20] Glucose: 182 (Spike)
[13:00] Glucose: 110
[19:00] Glucose: 100
[19:15] Meal: Shrimp stir-fry with rice (~65g carbs)
[19:15] Insulin: 6.5u rapid
[21:15] Glucose: 148

--- DAY -7 ---
[07:45] Glucose: 100
[08:00] Meal: 2 Eggs + 1 slice buttered toast (~15g carbs)
[08:00] Insulin: 1.5u rapid
[10:00] Glucose: 108
[14:00] Glucose: 92
[14:15] Meal: Homemade pesto pasta (~75g carbs)
[14:15] Insulin: 9u rapid
[16:00] Glucose: 65 (Hypo)
[16:05] Action: Consumed 15g fast-acting carbs (Apple juice)
[16:20] Glucose: 95

--- DAY -6 ---
[07:30] Glucose: 98
[07:45] Meal: Oatmeal + banana (~55g carbs)
[07:45] Insulin: 5u rapid
[09:45] Glucose: 177 (Spike)
[12:30] Glucose: 115
[19:00] Glucose: 110
[19:15] Meal: Pork chops and green beans (~20g carbs)
[19:15] Insulin: 2u rapid
[21:30] Glucose: 118

--- DAY -5 ---
[08:00] Glucose: 105
[08:15] Meal: 3 Scrambled eggs + avocado (~10g carbs)
[08:15] Insulin: 1u rapid
[10:15] Glucose: 110
[13:00] Glucose: 102
[13:15] Meal: Turkey sandwich on whole wheat (~45g carbs)
[13:15] Insulin: 4.5u rapid
[15:30] Glucose: 135
[19:45] Meal: Beef stew (~40g carbs)
[19:45] Insulin: 4u rapid
[22:00] Glucose: 125

--- DAY -4 ---
[07:50] Glucose: 102
[08:00] Meal: Oatmeal + banana (~55g carbs)
[08:00] Insulin: 5u rapid
[10:00] Glucose: 181 (Spike)
[13:30] Glucose: 112
[19:00] Glucose: 115
[19:15] Meal: Grilled salmon and asparagus (~12g carbs)
[19:15] Insulin: 1u rapid
[21:15] Glucose: 109

--- DAY -3 ---
[08:15] Glucose: 97
[08:30] Meal: Protein shake + almonds (~15g carbs)
[08:30] Insulin: 1.5u rapid
[10:30] Glucose: 104
[13:00] Glucose: 99
[13:15] Meal: Chicken fajita bowl with rice (~60g carbs)
[13:15] Insulin: 6u rapid
[15:30] Glucose: 142
[19:45] Meal: Chicken Caesar salad (~15g carbs)
[19:45] Insulin: 1.5u rapid
[22:00] Glucose: 114

--- DAY -2 ---
[07:45] Glucose: 104
[08:00] Meal: Oatmeal + banana (~55g carbs)
[08:00] Insulin: 5u rapid
[10:00] Glucose: 176 (Spike)
[13:00] Glucose: 118
[13:15] Meal: Lentil soup + side salad (~45g carbs)
[13:15] Insulin: 4.5u rapid
[15:30] Glucose: 125
[19:15] Meal: Steak and roasted broccoli (~15g carbs)
[19:15] Insulin: 1.5u rapid
[21:30] Glucose: 108

--- DAY -1 ---
[08:00] Glucose: 101
[08:15] Meal: 2 Eggs + sausage + 1 slice toast (~15g carbs)
[08:15] Insulin: 1.5u rapid
[10:15] Glucose: 111
[13:30] Glucose: 105
[13:45] Meal: Cobb salad (~15g carbs)
[13:45] Insulin: 1.5u rapid
[16:00] Glucose: 102
[19:15] Meal: Spaghetti bolognese (~65g carbs)
[19:15] Insulin: 6.5u rapid
[21:15] Glucose: 135
"""
print("Logs loaded.")

## 3. Run the Three Features

We define structured prompts for the three core patient-facing tools.

In [ ]:
def query_feature(system_instruction, user_content):
    if client is None:
        return "Client not configured. Please supply an API key."
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_content,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1
            )
        )
        return response.text
    except Exception as e:
        return f"Error generating content: {e}"

# Feature 1: Estimated Insulin Need
sys_f1 = f"""You are an expert AI endocrinology assistant.
Analyze the patient's demographics, regimen, and 10-day glucose logs to suggest the exact insulin dose for a new meal.
You must deduce the patient's Insulin-to-Carb Ratio (ICR) and Insulin Sensitivity Factor (ISF) from historical logs.
Respond ONLY with a JSON object containing keys: 'recommended_insulin' (float), 'reasoning' (string), and 'note' (string)."""

user_f1 = f"""Patient Metadata:
- Type: {DIABETES_TYPE}
- Regimen: {REGIMEN}
- Correction Target: {CORRECTION_TARGET}

Logs:
{LOGS_10D}

Current Context:
- Current Glucose: 130 mg/dL
- New Meal: Oatmeal + banana (~55g carbs)
- Last rapid insulin taken: 4 hours ago"""

print("============================================================")
print("FEATURE 1: Estimated Insulin Need")
print("============================================================")
print(query_feature(sys_f1, user_f1))

# Feature 2: Meal Impact Estimator
sys_f2 = """You are an expert AI endocrinology assistant.
Analyze the patient's logs to estimate how a specific planned meal will impact their glucose curve.
Respond ONLY with a JSON object containing keys: 'general_impact' (string), 'user_pattern' (string), and 'recommendation' (string)."""

user_f2 = f"""Logs:
{LOGS_10D}

Planned Meal:
- Avocado Toast & Eggs (~32g carbs, moderate fat & protein)"""

print("\n============================================================")
print("FEATURE 2: Meal Impact Estimator")
print("============================================================")
print(query_feature(sys_f2, user_f2))

# Feature 3: Health Insight (Reading Mode)
sys_f3 = """You are an expert AI endocrinology assistant.
Provide clinical context and actionable advice on the patient's recent glucose readings.
Respond ONLY with a JSON object containing keys: 'assessment' (string), 'context' (string), and 'action' (string)."""

user_f3 = f"""Logs:
{LOGS_10D}

Current Context:
- Current Glucose: 125 mg/dL
- Current Time: 13:00"""

print("\n============================================================")
print("FEATURE 3: Health Insight (Reading Mode)")
print("============================================================")
print(query_feature(sys_f3, user_f3))

## 4. Diabetic Assistant Chatbot

Here, we implement the interactive conversation session. It uses clinical guardrails to keep patient suggestions safe.

In [ ]:
def process_diabetic_query(user_message, raw_logs_10d, db_chat_history, diabetes_type, regimen):
    current_time_str = datetime.now().strftime("%A, %B %d, %Y, %I:%M %p")
    
    system_instruction = f"""
    CURRENT SYSTEM TIME: {current_time_str}

    [USER METADATA]
    DIABETES TYPE: {diabetes_type}
    MANAGEMENT REGIMEN: {regimen}

    You are an analytical, empathetic Diabetic Assistant Chatbot. You help users understand glucose patterns, meal impacts, and insulin timing. 
    
    CRITICAL MEDICAL ADAPTATION:
    Tailor your physiological reasoning, risk assessment, and behavioral advice strictly to the provided DIABETES TYPE and REGIMEN. 
    - If Type 1 (T1D): Prioritize meticulous insulin-to-carb calculation, insulin timing, insulin-on-board (stacking checks), and acute safety awareness.
    - If Type 2 (T2D): Prioritize evaluations of insulin resistance, impact of medication adherence, and lifestyle modifications.

    CRITICAL MEDICAL GUARDRAILS:
    1. Hypoglycemia (< 70 mg/dL): Always prioritize the 15-15 rule (15g fast-acting carbs, wait 15 mins).
    2. Mathematical Suggestions: You are encouraged to suggest specific insulin dose adjustments based on mathematical trends. 
    3. Mandatory Disclaimer: Every time you suggest a specific insulin dose or ratio, you MUST immediately include a clear, bolded warning stating that this is a mathematical estimation, not a medical prescription, and they must consult their doctor.
    4. Format: Be concise, use bullet points, and maintain a supportive tone.

    USER LOG CONTEXT:
    {raw_logs_10d}
    """
    
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.2
    )
    
    # Format history into SDK types
    formatted_history = []
    for msg in db_chat_history:
        text_content = msg["parts"][0] if isinstance(msg["parts"], list) else msg["parts"]
        formatted_history.append(
            types.Content(
                role=msg["role"],
                parts=[types.Part.from_text(text=text_content)]
            )
        )
        
    chat = client.chats.create(
        model="gemini-2.5-flash",
        history=formatted_history,
        config=config
    )
    
    response = chat.send_message(user_message)
    return response.text

# Test chatbot interaction
history = []
query = "Should I add more insulin tomorrow for this breakfast? 5 units isn't cutting it."

print("\n============================================================")
print(f"User: {query}")
print("============================================================")
reply = process_diabetic_query(query, LOGS_10D, history, DIABETES_TYPE, REGIMEN)
print(f"Assistant Output:\n {reply}")